# Honest Early-Warning Auditor

An early-warning detector is only useful if it fires on genuine pre-transition
segments **more often than its own false-alarm rate** on quiet, transition-free
segments. A raw detection rate hides that: a detector that alarms on 90% of events
but also on 90% of quiet nulls has learnt nothing.

`scpn_phase_orchestrator.evaluation.audit_detector` makes the honest comparison for
**any** detector from its per-segment scores alone: it calibrates a threshold to a
matched false-alarm rate, then tests whether the events alarm more than that by
chance with a label-permutation p-value, and seals the verdict into a hash-addressed
record. This notebook audits a skilful detector and a no-skill control on one corpus.

In [ ]:
import numpy as np
from scpn_phase_orchestrator.evaluation import audit_detector, seal_detector_audit

# A synthetic corpus of per-segment scores: 40 pre-transition *event* segments and
# 200 transition-free *null* segments. A skilful detector scores events higher than
# nulls; a no-skill detector cannot separate them.
rng = np.random.default_rng(0)
skilful_event_scores = rng.normal(2.0, 1.0, 40)   # events score high
null_scores = rng.normal(0.0, 1.0, 200)           # nulls score around zero
no_skill_event_scores = rng.normal(0.0, 1.0, 40)  # events indistinguishable from nulls
print(f"events: {len(skilful_event_scores)}   nulls: {len(null_scores)}")

## A skilful detector beats chance at a matched false alarm

The threshold is calibrated on the nulls to hold the false-alarm rate at the target
(0.10). The permutation p-value asks whether the events alarm *more than that matched
rate* by chance.

In [ ]:
skilful = audit_detector(
    event_scores=skilful_event_scores,
    null_scores=null_scores,
    detector_name="skilful-detector",
    target_false_alarm=0.10,
    n_permutations=2000,
)
print(f"matched threshold   : {skilful.matched_threshold:.3f}")
print(f"achieved false alarm: {skilful.achieved_false_alarm:.3f}  (target 0.10)")
print(f"detection rate      : {skilful.detection_rate:.3f}")
print(f"permutation p-value : {skilful.p_value:.4g}")
print(f"beats chance        : {skilful.beats_chance}")

## A no-skill detector lands at chance — and the auditor says so

The same corpus, a detector whose event scores are indistinguishable from the nulls.
Its detection rate sits near the false-alarm rate and the p-value is large: no skill.

In [ ]:
no_skill = audit_detector(
    event_scores=no_skill_event_scores,
    null_scores=null_scores,
    detector_name="no-skill-control",
    target_false_alarm=0.10,
    n_permutations=2000,
)
print(f"detection rate      : {no_skill.detection_rate:.3f}")
print(f"permutation p-value : {no_skill.p_value:.4g}")
print(f"beats chance        : {no_skill.beats_chance}")

assert skilful.beats_chance and not no_skill.beats_chance

## Seal the verdict into a tamper-evident record

`seal_detector_audit` binds the verdict to its corpus provenance under a SHA-256 over
its canonical JSON. Recomputing the hash from the recorded fields detects any later
edit, so a published audit cannot be quietly altered.

In [ ]:
record = seal_detector_audit(
    skilful,
    corpus_id="notebook-synthetic-corpus",
    captured_at="2026-07-07T00:00:00+00:00",
)
print(f"content hash: {record.content_hash}")
print(f"verifies    : {record.verify()}")
assert record.verify()

## Takeaway

The auditor is **detector-agnostic**: it reads per-segment scores, not a detector's
internals, so it judges the SCPN suite, an AR(1)/Kendall-τ baseline, or a black-box
classifier on identical footing. Point it at your own detector's scores with
`audit_detector`, or from the command line with `spo audit-detector scores.json`.
Turning “most detectors are at chance” from an embarrassment into a measurement is the
point.